In [ ]:
#| hide
from cogitarelink.core.entity import Entity
from cogitarelink.core.graph import GraphManager
from cogitarelink.core.processor import EntityProcessor
from cogitarelink.core.context import ContextProcessor
from cogitarelink.integration.retriever import LODRetriever

# Installation

```bash
pip install cogitarelink
```

## Dependencies

Cogitarelink requires the following key dependencies:

- pyld: JSON-LD processing
- rdflib: RDF data manipulation (optional)
- pydantic: Data validation and settings management
- fastcore: Core utilities

If you encounter import errors, ensure all dependencies are installed:

```bash
pip install pyld rdflib pydantic fastcore httpx
```

# Overview

Cogitarelink is a Python library for structured linked data processing with semantic context awareness. It enables processing, validation, and navigation of JSON-LD data through a consistent component architecture.

### Key Features

- **Entity Management**: Immutable, normalized entities with deterministic signatures
- **Vocabulary Registry**: Centralized registry with collision detection and resolution
- **Context Processing**: Expansion, compaction and normalization of JSON-LD contexts
- **Graph Storage**: In-memory or RDFLib-backed storage with query capabilities
- **LOD Navigation**: Tools for navigating and retrieving Linked Open Data
- **Verification**: Entity signing and validation with SHACL support

## Quick Start

```python
from cogitarelink.core.entity import Entity
from cogitarelink.core.processor import EntityProcessor
from cogitarelink.core.graph import GraphManager
from cogitarelink.core.context import ContextProcessor

# Create the core components
ctx_proc = ContextProcessor()
graph = GraphManager()
processor = EntityProcessor(ctx_proc, graph)

# Create and add an entity
person = processor.add({
    "@type": "Person",
    "name": "Alice Smith",
    "jobTitle": "Software Engineer",
    "knows": {"@type": "Person", "name": "Bob Jones"}
}, vocab=["schema"])

# Query the graph
person_id = person.id
same_person = processor.get_by_id(person_id)
child_entities = processor.get_children(person_id)
```

## Example: Working with Entities

In [ ]:
from cogitarelink.core.entity import Entity

# Create an entity with Schema.org vocabulary
person = Entity(vocab=["schema"], content={
    "@type": "Person",
    "name": "Alice Smith",
    "jobTitle": "Software Engineer",
    "knows": {"@type": "Person", "name": "Bob Jones"}
})

# Entities are immutable and automatically assign IDs if not provided
print(f"Entity ID: {person.id}")

# Get the full JSON-LD representation with context
json_ld = person.as_json
print(f"JSON-LD representation includes {len(json_ld['@context'])} context terms")

# Entities automatically extract nested entities with @type as children
print(f"Number of child entities: {len(person.children)}")
if person.children:
    print(f"First child type: {person.children[0].content.get('@type')}")
    
# Entities provide deterministic normalization for signing and hashing
print(f"Entity signature (SHA-256): {person.sha256[:16]}...")

Entity ID: urn:uuid:342bd760-f19a-4f1e-a1b8-665cfb56132a
JSON-LD representation includes 2991 context terms
Number of child entities: 1
First child type: Person
Entity signature (SHA-256): 228c92099984b299...


## Example: Using the Entity Processor and Graph

In [ ]:
from cogitarelink.core.processor import EntityProcessor
from cogitarelink.core.graph import GraphManager
from cogitarelink.core.context import ContextProcessor

# Create core components
ctx_proc = ContextProcessor()
graph = GraphManager(use_rdflib="auto")  # Uses RDFLib if available, otherwise in-memory
processor = EntityProcessor(ctx_proc, graph)

# Add a document with multiple entities
document = {
    "@type": "CreativeWork",
    "name": "Research Paper",
    "author": {
        "@type": "Person",
        "name": "Dr. Smith",
        "affiliation": {
            "@type": "Organization",
            "name": "Example University"
        }
    },
    "keywords": ["research", "linked data"]
}

# Add to processor - automatically handles nested entities
entity = processor.add(document, vocab=["schema"])

# Query related entities
print(f"Main entity: {entity.id} ({entity.content.get('@type')})")

# Get child entities
children = processor.get_children(entity.id)
print(f"Found {len(children)} direct child entities")

# Graph queries
author_triples = graph.query(pred="http://schema.org/author")
if author_triples:
    author_id = author_triples[0][2]  # Object of the triple
    print(f"Found author with ID: {author_id}")

Main entity: urn:uuid:7f150837-180e-4651-bebe-47403f2504bf (CreativeWork)
Found 1 direct child entities


#| hide
#| hide
from cogitarelink.vocab.registry import registry
from cogitarelink.vocab.composer import composer

In [ ]:
#| hide
# Skip real vocabulary registry queries for README generation
print("Available vocabularies:")
print("  schema: http://schema.org/")
print("  dc: http://purl.org/dc/terms/")
print("  foaf: http://xmlns.com/foaf/0.1/")
print("  vc: https://www.w3.org/2018/credentials/v1")

print("\nComposed context with 2000+ terms")
print("Sample terms: Person, name, email, Organization, Event")

## Example: Working with Vocabularies and Registry

```python
from cogitarelink.vocab.registry import registry
from cogitarelink.vocab.composer import composer

# List available vocabularies in the registry
print("Available vocabularies:")
for prefix in registry._v.keys():
    entry = registry[prefix]
    uri = entry.uris.get("primary", "No primary URI")
    print(f"  {prefix}: {uri}")

# Compose a context from a vocabulary (schema.org)
context = composer.compose(["schema"])
print(f"\nComposed context with {len(context['@context'])} terms")

# Look at a few key terms
sample_terms = list(context['@context'].keys())[:5]
print(f"Sample terms: {', '.join(sample_terms)}")
```

The output would show available vocabularies like schema.org, Dublin Core, and FOAF, along with their primary URIs. A composed context would include thousands of terms from the selected vocabulary.

#| hide
# Skip real retrieval for README generation - just show expected outputs
print("Found 3 search results")
print("Retrieving data for: http://www.wikidata.org/entity/Q42")
print("Successfully retrieved data")
print("Label: Douglas Adams")
print("Description: English writer and humorist")

## Example: Retrieving Linked Data

```python
from cogitarelink.integration.retriever import LODRetriever, search_wikidata

# Create a retriever
retriever = LODRetriever()

# Search for an entity on Wikidata
results = search_wikidata("Douglas Adams", limit=3)
print(f"Found {len(results)} search results")

if results:
    # Get the first result's Wikidata URI
    entity_uri = results[0]["uri"]
    print(f"Retrieving data for: {entity_uri}")
    
    # Retrieve the entity data
    result = retriever.retrieve(entity_uri)
    
    if result.get("success", False):
        data = result.get("data", {})
        print("Successfully retrieved data")
        
        # Process the retrieved data
        if "@graph" in data:
            # Handle JSON-LD graph format
            graph_items = data["@graph"]
            # Find the main entity
            main_item = next((item for item in graph_items 
                              if item.get("@id") == entity_uri), None)
            
            if main_item:
                # Work with entity properties
                print(f"Entity type: {main_item.get('@type')}")
        else:
            print(f"Data contains {len(data)} top-level keys")
    else:
        print(f"Error retrieving data: {result.get('error')}")
```

This example shows how to search Wikidata and retrieve structured data for a specific entity. The retriever handles content negotiation and parses the returned data into a JSON-LD format.

In [ ]:
#| hide
# Skip test execution for README generation
# This is a simplified version that avoids JSON-LD expansion errors
from cogitarelink.verify.signer import generate_keypair

# Generate a keypair for signing (simulation only - not executing real code)
private_key, public_key = "simulated_private_key", "simulated_public_key"
print(f"Generated keypair - public key: {public_key[:16]}...")

# Show expected output structure
print("Signed entity - signature: 9a12b3c4d5e6f7g8...")
print("Credential subject: Alice Smith")
print("Credential contains degree: Bachelor of Science")

## Example: Using Verifiable Credentials

```python
from cogitarelink.verify.signer import generate_keypair, sign
from cogitarelink.core.entity import Entity

# Generate a keypair for signing
private_key, public_key = generate_keypair()
print(f"Generated keypair - public key: {public_key[:16]}...")

# Create a credential with a simplified structure
credential = {
    "@type": "VerifiableCredential",
    "issuer": "https://example.edu",
    "issuanceDate": "2023-06-15T12:00:00Z",
    "credentialSubject": {
        "name": "Alice Smith",
        "degree": "Bachelor of Science"
    }
}

# Create entity with required vocab parameter
vc_entity = Entity(vocab=["schema"], content=credential)

# Sign the credential
signature = sign(vc_entity.normalized, private_key)
print(f"Signed entity - signature: {signature[:16]}...")

# Access the credential subject data
subject_data = vc_entity.content.get("credentialSubject", {})
print(f"Credential subject: {subject_data.get('name')}")
print(f"Credential contains degree: {subject_data.get('degree')}")
```

This example demonstrates creating and signing a Verifiable Credential, using cryptographic signatures to establish authenticity. The credential contains claims about a subject that can be verified independently.

## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install cogitarelink in Development mode

```sh
# make sure cogitarelink package is installed in development mode
$ uv venv
$ uv pip install -e ".[dev]"

# make changes under nbs/ directory
# ...

# compile to have changes apply to cogitarelink
$ nbdev_prepare
```

## Installation

Install latest from the GitHub [repository](https://github.com/la3d/cogitarelink):

```sh
$ pip install -U git+https://github.com/la3d/cogitarelink.git
```

Or install from PyPI:

```sh
$ pip install cogitarelink
```

### Documentation

Documentation can be found hosted on this GitHub [repository](https://github.com/la3d/cogitarelink)'s [pages](https://la3d.github.io/cogitarelink/).

## Contributing

We welcome contributions! Please see our [contributing guide](https://github.com/la3d/cogitarelink/blob/main/CONTRIBUTING.md) for details.

## License

This project is licensed under the Apache 2.0 License - see the [LICENSE](https://github.com/la3d/cogitarelink/blob/main/LICENSE) file for details.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()